In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import polars as pl
from influxdb_client  import InfluxDBClient , WriteOptions
import yaml
import requests


current_file = Path(globals().get('__vsc_ipynb_file__', None))
parent_dir = current_file.parent.parent
os.chdir(parent_dir)
print("Changed working directory to:", Path.cwd())

from dotenv import load_dotenv
load_dotenv("mlops_services/.env.mlops")

Changed working directory to: /home/legacy/Projects/Weather_forecasting_with_MLOps


True

In [12]:
from mlops_services.src.utils.time_utils import convert_to_utc,include_time

# convert_to_utc('2023-01-01 00:00','2025-06-30 23:00', tz="Asia/Kolkata",ts_format="%Y-%m-%d %H:%M")

In [43]:
%run mlops_services/src/pipelines/train/01_retrieve.py db BLR "2023-01-01 00:00" "2025-06-30 23:00"

('2023-01-01 00:00', '2025-06-30 23:00')
InfluxDB connection successful.
loading historic data from DB


In [39]:
from mlops_services.src.utils import command_line, load_config,time_utils
parser = command_line.Args(prog='DataRetrieval')
parser.add_argument("source", help="db/local", type=str)
parser.add_argument("city", help="City", type=str)
parser.add_argument("start_time", help="Start time", type=str)
parser.add_argument("end_time", help="End time", type=str)


_StoreAction(option_strings=[], dest='end_time', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=True, help='End time', metavar=None)

In [10]:
%tb

TypeError: module() takes at most 2 arguments (3 given)

In [44]:
df_train = pl.read_parquet('mlops_services/data/raw/test.parquet')
df_train

result,table,_start,_stop,_time,_measurement,city,cloud_cover,dew_point_2m,elevation,et0_fao_evapotranspiration,generationtime_ms,is_day,latitude,longitude,rain,relative_humidity_2m,surface_pressure,temperature_2m,time_local,timezone,timezone_abbreviation,utc_offset_seconds,vapour_pressure_deficit,weather_code,wind_direction_10m,wind_gusts_10m,wind_speed_10m
str,i64,"datetime[ns, UTC]","datetime[ns, UTC]","datetime[ns, UTC]",str,str,i64,f64,f64,f64,f64,i64,f64,f64,f64,i64,f64,f64,str,str,str,i64,f64,i64,i64,f64,f64
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2024-11-14 18:30:00 UTC,"""city_BLR""","""BLR""",100,19.6,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,98,912.6,19.9,"""2024-11-15 00:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.05,3,69,22.7,10.6
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2024-11-14 19:30:00 UTC,"""city_BLR""","""BLR""",100,19.5,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,98,912.0,19.8,"""2024-11-15 01:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.04,3,75,22.7,9.5
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2024-11-14 20:30:00 UTC,"""city_BLR""","""BLR""",100,19.2,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,96,911.4,19.9,"""2024-11-15 02:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.1,3,77,22.7,9.6
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2024-11-14 21:30:00 UTC,"""city_BLR""","""BLR""",100,19.1,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,95,911.3,20.0,"""2024-11-15 03:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.12,3,73,22.7,9.6
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2024-11-14 22:30:00 UTC,"""city_BLR""","""BLR""",100,19.1,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,95,911.3,19.8,"""2024-11-15 04:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.11,3,78,22.3,10.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2025-06-30 13:30:00 UTC,"""city_BLR""","""BLR""",100,18.5,918.0,0.07,8.804321,0,12.970123,77.56364,0.1,72,908.4,23.9,"""2025-06-30 19:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.84,51,268,34.2,11.7
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2025-06-30 14:30:00 UTC,"""city_BLR""","""BLR""",100,18.8,918.0,0.04,8.804321,0,12.970123,77.56364,0.1,75,909.0,23.4,"""2025-06-30 20:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.71,51,254,31.3,14.8
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 17:30:01 UTC,2025-06-30 15:30:00 UTC,"""city_BLR""","""BLR""",99,18.7,918.0,0.04,8.804321,0,12.970123,77.56364,0.0,78,909.4,22.6,"""2025-06-30 21:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.59,3,260,36.4,15.9
